# KITP Effect of Future Arctic Sea Ice Loss on the Greenland Ice Sheet 

In [ ]:
from functools import partial
from pathlib import Path
import json                                                                                                                                                                        
import dask
import xarray as xr
import pint_xarray
import matplotlib.pylab as plt
import matplotlib as mpl
import nc_time_axis
import numpy as np
import pandas as pd
from dask.diagnostics import ProgressBar
from pism_terra.processing import preprocess_netcdf as preprocess
import cartopy.crs as ccrs 
import warnings                                                                                                                                                    
warnings.filterwarnings("ignore", message="Increasing number of chunks")
from cycler import cycler
import cmweather
from cmap import Colormap
cm = Colormap('tol:bright').to_matplotlib()
cm_cycler = cycler(color=cm.colors) 
cm_precip = Colormap("crameri:navia").to_matplotlib()
cm_rdbu = Colormap("cmocean:balance").to_matplotlib()

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
delta_coder = xr.coders.CFTimedeltaCoder()

## Ice-sheet wide scalars

In [ ]:
data_dir = "/Volumes/LHS800DATA"
exps = {
        'pdSST-futArcSICSIT_pdSST-pdSICSIT': {'short_hand': 'pdSST-futArcSICSIT_pdSST-pdSICSIT', 'color': (0.0660, 0.4430, 0.7450), 'ls': "solid"},
        'pa-futArcSIC-ext_pa-pdSIC-ext': {'short_hand': 'pa-futArcSIC-ext_pa-pdSIC-ext', 'color': (0.8660, 0.3290, 0), 'ls': "solid"},
        'futSST-pdSIC_pdSST-pdSIC': {'short_hand': 'futSST-pdSIC_pdSST-pdSIC', 'color': (0.9290, 0.6940, 0.1250), 'ls': "solid"},
        'pdSST-futArcSIC_pdSST-pdSIC': {'short_hand': 'pdSST-futArcSIC_pdSST-pdSIC', 'color': (0.5210, 0.0860, 0.8190), 'ls': "solid"},
        'HIRHAM5-ERA5_YMM_1990_2019': {'short_hand': 'baseline', 'color': (0 , 0, 0), 'ls': "dashed"},
}
gt2mmsle = xr.DataArray(-1 / 361.8).pint.quantify("mm/Gt")

In [ ]:
%%time
scalar_files = list(Path(f"{data_dir}/2026_03_kitp_v3/output/scalar/").glob("scalar_*1200m*.nc"))
scalar_lowres_files = list(Path(f"{data_dir}/2026_03_kitp_v3/output/scalar/").glob("scalar_*1200m*.nc"))
scalar_files = list(Path(f"{data_dir}/2026_03_kitp_v3/output/spatial/").glob("fldsum_*2400m*.nc"))

def load_dataset(filename_or_obj: str | Path, **kwargs) -> xr.Dataset:
    with ProgressBar():
        ds = xr.open_mfdataset(filename_or_obj,
                               preprocess=partial(preprocess, **kwargs), 
                               engine="netcdf4",
                               parallel=True,
                               chunks=None,
                               data_vars="all",
                               join="outer",
                               decode_times=time_coder,
                               decode_timedelta=delta_coder)
    ds["exp_id"] = ds["exp_id"].str.replace("CESM1-WACCM-SC_", "") 
 
    return ds.resample(time="YE").mean()

with dask.config.set(**{"array.slicing.split_large_chunks": False}):
    scalar = load_dataset(scalar_files,
                                 exp_regexp=r"id_(.+?)_0",                                                                                                                     
                                 uq_regexp=None,   
                                 uq_dim=None,
                                 exp_dim="exp_id").sel({"exp_id": list(exps.keys())}).pint.quantify()
    # try:
    #     scalar = scalar.sel(basin="GIS").squeeze()
    # except:
    #     pass

    scalar_lowres = load_dataset(scalar_lowres_files,
                                 exp_regexp=r"id_(.+?)_0",                                                                                                                     
                                 uq_regexp=None,   
                                 uq_dim=None,
                                 exp_dim="exp_id").sel({"exp_id": list(exps.keys())}).pint.quantify()
    try:
        scalar_lowres = scalar_lowres.sel(basin="GIS").squeeze()
    except:
        pass

In [ ]:
pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

## Plot ice sheet-wide scalars

In [ ]:

config = json.loads(scalar.pism_config.sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019").item())
res = f"{int(config["grid.dx"])}m"

with mpl.rc_context(rc=rc_params):
    fig, axs = plt.subplots(2, 1, sharex=True, figsize=(6.4, 3.6), height_ratios=[2, 1])
    for exp, _ds in scalar.groupby("exp_id"):
        ice_mass = _ds["ice_mass"].pint.to("Gt").pint.dequantify()
        ice_mass -= ice_mass.isel({"time": 0})
        slc = ice_mass * gt2mmsle
        glf = _ds["grounding_line_flux"].pint.to("Gt/yr")
        
        slc.plot(ax=axs[0], color=exps[exp]["color"], ls=exps[exp]["ls"], label=exps[exp]["short_hand"], lw=1)
        glf.plot(ax=axs[1], color=exps[exp]["color"], ls=exps[exp]["ls"], lw=1)
    for exp, _ds in scalar_lowres.groupby("exp_id"):
        ice_mass = _ds["ice_mass"].pint.to("Gt").pint.dequantify()
        ice_mass -= ice_mass.isel({"time": 0})
        slc = ice_mass * gt2mmsle
        glf = _ds["grounding_line_flux"].pint.to("Gt/yr")
        
        slc.plot(ax=axs[0], color=exps[exp]["color"], ls=exps[exp]["ls"], lw=0.25)
        glf.plot(ax=axs[1], color=exps[exp]["color"], ls=exps[exp]["ls"], lw=0.25)
        
    axs[0].set_ylabel("Contribution to sea-level (mm))")
    axs[0].set_xlabel(None)
    axs[1].set_ylabel("Grounding line flux (Gt/yr)")
    axs[0].set_title(None)
    axs[1].set_title(None)
    # # Create a legend outside the figure at the bottom middle
    handles, labels = axs[0].get_legend_handles_labels()
    legend_main = fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.25, 0.70), ncol=1)
    legend_main.set_title(None)
    legend_main.get_frame().set_linewidth(0.0)
    legend_main.get_frame().set_alpha(0.0)
    fig.tight_layout()
    plt.show()
    fig.savefig(f"pism_kitp_{res}.png", dpi=300)
    fig.savefig(f"pism_kitp_{res}.pdf")
    plt.close()
    del fig


## Plot basin-wide cumulative mass change

In [ ]:

config = json.loads(scalar.pism_config.sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019").item())
res = f"{int(config["grid.dx"])}m"


with mpl.rc_context(rc=rc_params):
    for basin_name, basin in scalar.groupby("basin"):
        fig, ax = plt.subplots(1, 1, sharex=True, figsize=(6.4, 3.6))
        for exp, _ds in basin.groupby("exp_id"):
            ice_mass = _ds["ice_mass"].pint.to("Gt").pint.dequantify()
            ice_mass -= ice_mass.isel({"time": 0})
            slc = ice_mass * gt2mmsle
            
            slc.plot(ax=ax, color=exps[exp]["color"], ls=exps[exp]["ls"], label=exps[exp]["short_hand"], lw=1)
            
        ax.set_ylabel("Contribution to sea-level (mm))")
        ax.set_xlabel(None)
        ax.set_title(None)
        # # Create a legend outside the figure at the bottom middle
        handles, labels = ax.get_legend_handles_labels()
        legend_main = fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.25, 0.70), ncol=1)
        legend_main.set_title(None)
        legend_main.get_frame().set_linewidth(0.0)
        legend_main.get_frame().set_alpha(0.0)
        fig.tight_layout()
        plt.show()
        fig.savefig(f"pism_kitp_{res}_{basin_name}.png", dpi=300)
        fig.savefig(f"pism_kitp_{res}_{basin_name}.pdf")
        plt.close()
        del fig


In [ ]:
baseline_climate_file = Path('/media/andy/LHS800DATA/2026_03_kitp_1500m/kitp/input/v3/HIRHAM5-ERA5_YMM_1990_2019_v3.nc')
anomalies_files = list(Path("/media/andy/LHS800DATA/2026_03_kitp_1500m/kitp/input/v3/").glob("CESM1*.nc"))
climate_files = list(Path("/media/andy/LHS800DATA/2026_03_kitp_1500m/kitp/input/v3/").glob("CESM1*.nc"))

In [ ]:
with ProgressBar():
    baseline = xr.open_dataset(baseline_climate_file, engine="netcdf4",
                               decode_times=time_coder,
                               decode_timedelta=delta_coder)
    anomalies = xr.open_mfdataset(anomalies_files,
                                  preprocess=partial(preprocess, 
                                                     exp_regexp=r"_anomalies_(.+?)_v\d+\.nc",                                                                                                              
                                                     exp_dim="gcm_id",
                                                     process_config=False), 
                                  engine="netcdf4",
                                  parallel=True,
                                  chunks=None,
                                  data_vars="minimal",
                                  join="outer",
                                  decode_times=time_coder,
                                  decode_timedelta=delta_coder)

In [ ]:
anomalies_summer_mean = anomalies.isel(time=[6, 7, 8]).mean(dim="time").thin({"x": 5, "y": 5})
anomalies_winter_mean = anomalies.isel(time=[0, 1, 2]).mean(dim="time").thin({"x": 5, "y": 5})

In [ ]:
with mpl.rc_context(rc=rc_params):

    fg = anomalies_summer_mean.air_temp.plot(                                                                                                                     
        col="gcm_id",
        col_wrap=3,                                                                                                                                                          cmap=cm_rdbu,                                         
        figsize=(6.4, 6.4),             
        cbar_kwargs={"shrink": 0.6,},                                                                                                                    
    )
    for ax, gcm in zip(fg.axs.flat, anomalies_summer_mean.gcm_id.values):
        ax.set_title(str(gcm), fontsize=6)
    fig = fg.fig
    fig.savefig("kitp_air_temp_jja_mean.png", dpi=300)
    
    fg = anomalies_winter_mean.precipitation.plot(                                                                                                                     
        col="gcm_id",
        col_wrap=3,                                                                                                                                                          cmap=cm_rdbu,                                         
        figsize=(6.4, 6.4),             
        cbar_kwargs={"shrink": 0.6,},                                                                                                                    
    )
    for ax, gcm in zip(fg.axs.flat, anomalies_winter_mean.gcm_id.values):
        ax.set_title(str(gcm), fontsize=6)
    fig = fg.fig
    fig.savefig("kitp_precipitation_jfm_mean.png", dpi=300)  
 

In [ ]:
%%time
spatial_lowres_files = list(Path("/media/andy/LHS800DATA/2026_03_kitp_1500m/output/spatial/").glob("clipped_*.nc"))

def load_dataset(filename_or_obj: str | Path) -> xr.Dataset:
    with ProgressBar():
        ds = xr.open_mfdataset(filename_or_obj,
                                      preprocess=partial(preprocess, 
                                                         exp_regexp=r"id_(.+?)_uq_",                                                                                                                     
                                                         uq_regexp="uq_(.+?)_",   
                                                         uq_dim="uq_id",
                                                         exp_dim="exp_id"), 
                                      engine="netcdf4",
                                      parallel=True,
                                      chunks="auto",
                                      data_vars="minimal",
                                      join="outer",
                                      decode_times=time_coder,
                                      decode_timedelta=delta_coder).pint.quantify()
    ds["exp_id"] = ds["exp_id"].str.replace("CESM1-WACCM-SC_", "") 
 
    return ds

with dask.config.set(**{"array.slicing.split_large_chunks": False}):
    spatial_lowres = load_dataset(spatial_lowres_files).sel({"exp_id": exps})

In [ ]:
speed = spatial_lowres.isel({"time": [0, -1]})["velsurf_mag"].quantile(0.50, dim="uq_id", skipna=True)

In [ ]:
fg = speed.plot(col="exp_id", row="time")
fig = fg.fig
fig.savefig("kitp_speed_median_300yr.png", dpi=300)  

In [ ]:
scalar.sel({"pism_config_axis": ["grid.dx"]})

In [ ]:
scalar.pism_config_axis

In [ ]:
_ice_mass.time

In [ ]:
scalar_hires.rename_vars?

In [ ]:
scalar

In [ ]:
scalar_lowres

In [ ]:
_grounding_line_flux.values

In [ ]:
scalar_lowres_files

In [ ]:
ice_mass.plot(hue="exp_id")
plt.show()

In [ ]:
scalar_lowres["ice_mass"].values

In [ ]:
        ice_mass = (scalar_lowres["ice_mass"].pint.to("Gt") * gt2mmsle).pint.dequantify()
        #ice_mass = ice_mass.isel({"time": 0})


In [ ]:
_ice_mass

In [ ]:

_surface_flux.values

In [ ]:
cm

In [ ]:
ice_mass.time

In [ ]:
_grounding_line_flux.values

In [ ]:
_ds.grounding_line_flux.values

In [ ]:
ice_mass.sel({"exp_id": "HIRHAM5-ERA5_YMM_1990_2019"}).squeeze()

In [ ]:
ice_mass.values

In [ ]:
scalar = load_dataset(scalar_files,
                                 exp_regexp=r"id_(.+?)_0",                                                                                                                     
                                 uq_regexp=None,   
                                 uq_dim=None,
                                 exp_dim="exp_id",
                                 process_config=True)

In [ ]:
scalar.pism_config

In [ ]:
preprocess?

In [ ]:
  import netCDF4                                                                                                                                                                     
  nc = netCDF4.Dataset(scalar_files[0])
  print("pism_config" in nc.variables)
  nc.close() 

In [ ]:
scalar_files

In [ ]:
ds = xr.open_dataset(scalar_files[0])

In [ ]:
pds = preprocess(ds,                                 
           exp_regexp=r"id_(.+?)_0",                                                                                                                     
           uq_regexp=None,   
           uq_dim=None,
           exp_dim="exp_id",
          )

In [ ]:
scalar.time